Using multiple supervision heads to disentangle different covaraites in the latent space.

In [52]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import pyrootutils
root = pyrootutils.setup_root(
    search_from=".",
    indicator=".git",
    pythonpath=True,
    dotenv=True,
)

import torch
import pytorch_lightning as pl
import numpy as np
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt     
import seaborn as sns
import pandas as pd
from experiments.src.utils import load_and_preprocess
from scdeepsim.truncated_normal_vae import TruncatedNormalVAE
from scdeepsim.dataset import ScDataModule
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder

In [54]:
embryo_atlas_dir = root / "data" / "HVG_embryoatlas.h5ad"
n_cells = 10000
n_genes = 2000
embryo_atlas = load_and_preprocess(embryo_atlas_dir, n_cells, n_genes)
embryo_atlas.obs.head()

/opt/miniconda3/envs/lightning/lib/python3.12/site-packages/scanpy/preprocessing/_highly_variable_genes.py:178: Trying to modify attribute `._uns` of view, initializing view as actual.


,cell,barcode,sample,pool,stage,sequencing.batch,theiler,doub.density,doublet,cluster,cluster.sub,cluster.stage,cluster.theiler,stripped,celltype,colour,sizeFactor,n_genes
cell_57591,cell_57591,ACGATTCTGGGAGT,22,17,mixed_gastrulation,2,TS9-10,0.023555,False,19,3,5,5,False,Blood progenitors 1,f9decf,0.697793,156
cell_53017,cell_53017,ACGTCAGATGAACC,21,17,mixed_gastrulation,2,TS9-10,0.233501,False,17,6,7,7,False,Mesenchyme,cc7818,0.862364,187
cell_133473,cell_133473,GCAGCTCTTCTAGG,36,25,E8.5,3,TS12,0.396087,False,17,4,1,16,False,Mesenchyme,cc7818,1.662080,518
cell_105175,cell_105175,CTAGATCTAGCATC,30,23,E7.0,3,TS10,2.236907,False,1,7,5,3,False,Epiblast,635547,1.891389,347
cell_4390,cell_4390,GCACACCTTGGCAT,6,6,E7.5,1,TS11,0.318799,False,4,8,6,15,False,ExE endoderm,7F6874,0.898551,268


In [55]:
embryo_atlas.obs['sequencing.batch'] = embryo_atlas.obs['sequencing.batch'].astype('category')
embryo_atlas.obs.rename(columns={'sequencing.batch': 'batch'}, inplace=True)
n_celltypes = len(embryo_atlas.obs['celltype'].unique())
print(f"  Found {n_celltypes} unique celltypes")
penalty_weight = 5.0

celltype_sup_head_config = {
    "name": "celltype",
    "type": "categorical",
    "n_classes": n_celltypes,
    "latent_dims": 16,
    "weight": penalty_weight}

n_batches = len(embryo_atlas.obs['batch'].unique())
print(f"  Found {n_batches} unique batches")
batch_sup_head_config = {
    "name": "batch",
    "type": "categorical",
    "n_classes": n_batches,
    "latent_dims": 16,
    "weight": penalty_weight}

n_stages = len(embryo_atlas.obs['stage'].unique())
print(f"  Found {n_stages} unique stages")
stage_sup_head_config = {
    "name": "stage",
    "type": "categorical",
    "n_classes": n_stages,
    "latent_dims": 16,
    "weight": penalty_weight}

sup_config = [celltype_sup_head_config, batch_sup_head_config, stage_sup_head_config]

  Found 37 unique celltypes
  Found 3 unique batches
  Found 10 unique stages


In [56]:
def train_and_evaluate_classifier(X_train, y_train, X_test, y_test, name, seed):
    clf = RandomForestClassifier(
        random_state=seed,
    )
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)

    print(f"  {name}:")
    print(f"    Accuracy:          {acc:.4f}")
    print(f"    Balanced Accuracy: {bal_acc:.4f}")

    return acc, bal_acc

celltype_le = LabelEncoder()
celltype_labels = celltype_le.fit_transform(embryo_atlas.obs["celltype"])
batch_le = LabelEncoder()
batch_labels = batch_le.fit_transform(embryo_atlas.obs["batch"])
stage_le = LabelEncoder()
stage_labels = stage_le.fit_transform(embryo_atlas.obs["stage"])

train_idx, test_idx = train_test_split(
    np.arange(len(celltype_labels)),
    test_size=0.2,
    random_state=42,
    stratify=celltype_labels,
)
celltype_y_train, celltype_y_test = celltype_labels[train_idx], celltype_labels[test_idx]
batch_y_train, batch_y_test = batch_labels[train_idx], batch_labels[test_idx]
stage_y_train, stage_y_test = stage_labels[train_idx], stage_labels[test_idx]



In [ ]:
vae = TruncatedNormalVAE(
    n_genes=n_genes,
    latent_dim=128,
    enc_hidden=[512, 256],
    dec_hidden=[256, 512],
    supervised_config=sup_config,
    sup_head_hidden=64,
)

data_module = ScDataModule(
    embryo_atlas,
    label_keys={
        "celltype": {"obs_key": "celltype", "type": "categorical"},
        "batch": {"obs_key": "batch", "type": "categorical"},
        "stage": {"obs_key": "stage", "type": "categorical"},
    },
)

trainer = pl.Trainer(
    max_epochs=50,
    accelerator="auto",
    devices="auto",
    log_every_n_steps=50,
    enable_checkpointing=True,
    gradient_clip_val=vae.gradient_clip_val,
)

trainer.fit(vae, data_module)

In [58]:
device = next(vae.parameters()).device
X_log1p = torch.tensor(embryo_atlas.X, dtype=torch.float32, device=device)

vae.eval()
with torch.no_grad():
    mu_z, logvar_z = vae.encode(X_log1p)
    z_original = vae.reparameterize(mu_z, logvar_z)

    z_ablated = z_original.clone()
    celltype_slice = vae._sup_slices.get("celltype", slice(0, 0))
    z_celltype = z_original[:, celltype_slice].clone()
    z_ablated[:, celltype_slice] = 0.0

    print(f"  Ablated latent dims {celltype_slice.start}:{celltype_slice.stop}")

    batch_slice = vae._sup_slices.get("batch", slice(0, 0))
    z_batch = z_original[:, batch_slice].clone()
    z_ablated[:, batch_slice] = 0.0

    print(f"  Ablated latent dims {batch_slice.start}:{batch_slice.stop}")

    stage_slice = vae._sup_slices.get("stage", slice(0, 0))
    z_stage = z_original[:, stage_slice].clone()
    z_ablated[:, stage_slice] = 0.0

    print(f"  Ablated latent dims {stage_slice.start}:{stage_slice.stop}")

    decoded_original = vae.sample_from_latent(z_original).cpu().numpy()
    decoded_ablated = vae.sample_from_latent(z_ablated).cpu().numpy()

  Ablated latent dims 0:16
  Ablated latent dims 16:32
  Ablated latent dims 32:48


In [59]:


print("Random Chance:")
print(f"Celltype: {1.0 / n_celltypes:.4f}")
print(f"Batch: {1.0 / n_batches:.4f}")
print(f"Stage: {1.0 / n_stages:.4f}")

print("Cell Type Classification")
train_and_evaluate_classifier(
    embryo_atlas.X[train_idx], celltype_y_train, 
    embryo_atlas.X[test_idx], celltype_y_test, "Celltype Classification on Real Data", 42
)

train_and_evaluate_classifier(
    z_celltype[train_idx], celltype_y_train,
z_celltype[test_idx], celltype_y_test, "Celltype Classification on Celltype Latent", 42
)

# celltype classification on batch latent
train_and_evaluate_classifier(
    z_batch[train_idx], celltype_y_train,
    z_batch[test_idx], celltype_y_test, "Celltype Classification on Batch Latent", 42
)


print("Batch Classification")
# batch classification on real data
train_and_evaluate_classifier(
    embryo_atlas.X[train_idx], batch_y_train,
    embryo_atlas.X[test_idx], batch_y_test, "Batch Classification on Real Data", 42
)

# batch classification on celltype latent
train_and_evaluate_classifier(
    z_celltype[train_idx], batch_y_train,
    z_celltype[test_idx], batch_y_test, "Batch Classification on Celltype Latent", 42
)

# batch classification on batch latent
train_and_evaluate_classifier(
    z_batch[train_idx], batch_y_train,
    z_batch[test_idx], batch_y_test, "Batch Classification on Batch Latent", 42
)

print("Stage Classification")
# stage classification on real data
train_and_evaluate_classifier(
    embryo_atlas.X[train_idx], stage_y_train,
    embryo_atlas.X[test_idx], stage_y_test, "Stage Classification on Real Data", 42
)

# stage classification on celltype latent
train_and_evaluate_classifier(
    z_celltype[train_idx], stage_y_train,
    z_celltype[test_idx], stage_y_test, "Stage Classification on Celltype Latent", 42
)

# stage classification on batch latent
train_and_evaluate_classifier(
    z_batch[train_idx], stage_y_train,
    z_batch[test_idx], stage_y_test, "Stage Classification on Batch Latent", 42
)

# stage classification on stage latent
train_and_evaluate_classifier(
    z_stage[train_idx], stage_y_train,
    z_stage[test_idx], stage_y_test, "Stage Classification on Stage Latent", 42
)

Random Chance:
Celltype: 0.0270
Batch: 0.3333
Stage: 0.1000
Cell Type Classification
  Celltype Classification on Real Data:
    Accuracy:          0.7435
    Balanced Accuracy: 0.5966
  Celltype Classification on Celltype Latent:
    Accuracy:          0.8450
    Balanced Accuracy: 0.7633
  Celltype Classification on Batch Latent:
    Accuracy:          0.1295
    Balanced Accuracy: 0.0355
Batch Classification
  Batch Classification on Real Data:
    Accuracy:          0.6325
    Balanced Accuracy: 0.4289
  Batch Classification on Celltype Latent:
    Accuracy:          0.5850
    Balanced Accuracy: 0.3744
  Batch Classification on Batch Latent:
    Accuracy:          0.5730
    Balanced Accuracy: 0.3553
Stage Classification
  Stage Classification on Real Data:
    Accuracy:          0.4650
    Balanced Accuracy: 0.3720
  Stage Classification on Celltype Latent:
    Accuracy:          0.3675
    Balanced Accuracy: 0.2825
  Stage Classification on Batch Latent:
    Accuracy:          0

(0.5525, 0.4384859530159712)